# Notebook 17 — Embeddings: Teaching a Computer What "Similar" Means
**AI Models and Agents for Business · Week 9 · Session 17**

Today's lecture built the RAG pipeline as a diagram: chunk → embed → store → retrieve → generate. This notebook makes the **embed** and **retrieve** steps concrete, so that when you build the full pipeline next session on a real 10-K, nothing in it is a black box.

**No API key needed.** We use `sentence-transformers` locally — the same embedding model Notebook 18 uses next session.

| Part | You'll do |
|---|---|
| A | Turn sentences into vectors and look at them |
| B | Measure meaning with cosine similarity |
| C | Race semantic search against keyword search |
| D | Discover the chunk-size tradeoff before it bites you |

⚠️ **Cell 1 downloads an embedding model (~90 MB).** Run it as soon as you sit down.

## Cell 1 — Setup

In [ ]:
!pip install sentence-transformers --quiet

import numpy as np
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded: all-MiniLM-L6-v2")
print(f"Output dimensions: {embedder.get_sentence_embedding_dimension()}")

---
## Part A: A sentence becomes 384 numbers

An **embedding** is a list of numbers representing a piece of text's *meaning*. Similar meanings land near each other in that 384-dimensional space, even when the words are completely different.

In [ ]:
sentence = "Our revenue grew 12% last quarter."
vec = embedder.encode([sentence])[0]

print(f'Sentence : "{sentence}"')
print(f"Vector length : {len(vec)}")
print(f"First 8 values: {np.round(vec[:8], 4)}")
print(f"Vector norm    : {np.linalg.norm(vec):.4f}")
print()
print("That's it. Every document in every RAG system is stored as a list like this.")
print("The numbers are not human-readable on their own — their meaning lives in")
print("how CLOSE they are to other vectors, which is what Part B measures.")

## Cell 3 — The key demonstration: same meaning, different words

Watch these two pairs. One pair shares almost no words but means the same thing. The other shares most of its words but means something different.

In [ ]:
pair_a = ("Our revenue grew 12% last quarter.",
          "Sales increased twelve percent in Q3.")
pair_b = ("Our revenue grew 12% last quarter.",
          "Our revenue fell 12% last quarter.")

def similarity(s1, s2):
    e = embedder.encode([s1, s2])
    return float(cosine_similarity([e[0]], [e[1]])[0][0])

def word_overlap(s1, s2):
    w1 = set(s1.lower().replace('.','').replace('%','').split())
    w2 = set(s2.lower().replace('.','').replace('%','').split())
    return len(w1 & w2) / len(w1 | w2)

for label, (s1, s2) in [("SAME meaning, different words", pair_a),
                        ("SIMILAR words, opposite meaning", pair_b)]:
    print(f"{label}")
    print(f'  A: "{s1}"')
    print(f'  B: "{s2}"')
    print(f"  Word overlap      : {word_overlap(s1, s2):.0%}")
    print(f"  Embedding similarity: {similarity(s1, s2):.3f}")
    print()

### 💬 Discussion — Part A
- The first pair shares very few words but scores high on embedding similarity. A keyword search would have missed the connection entirely. Why does that matter for a company search tool?
- The second pair is the uncomfortable one. "grew 12%" and "fell 12%" are *opposites*, yet the embedding similarity is high because the sentences are structurally and topically near-identical. What does that tell you about what embeddings actually capture?
- **Business risk:** a RAG system retrieves a chunk saying revenue *fell* when the user asked why it *grew*. The retrieval looked successful by similarity score. How would you catch this before a customer does?

---
## Part B: A similarity map of a company's documents

Now let's embed a set of business sentences from different domains and look at the full similarity matrix. Related topics should cluster.

In [ ]:
docs = [
    "Q3 revenue increased 12% year over year.",           # finance
    "Sales for the third quarter rose twelve percent.",   # finance (paraphrase)
    "Operating margin compressed due to input costs.",    # finance
    "The customer reported a billing error on their invoice.",  # support
    "A client complained about an incorrect charge.",      # support (paraphrase)
    "Our new hire onboarding takes six weeks.",            # HR
    "Employee ramp-up time is approximately 1.5 months.",  # HR (paraphrase)
    "The server experienced downtime on Tuesday night.",   # engineering
]
short_labels = ["FIN revenue+12%", "FIN sales+12%", "FIN margin", "SUP billing error",
                "SUP wrong charge", "HR onboarding 6wk", "HR ramp 1.5mo", "ENG downtime"]

emb = embedder.encode(docs)
sim = cosine_similarity(emb)

fig, ax = plt.subplots(figsize=(9, 7.5))
im = ax.imshow(sim, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(docs))); ax.set_xticklabels(short_labels, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(docs))); ax.set_yticklabels(short_labels, fontsize=8)
for i in range(len(docs)):
    for j in range(len(docs)):
        ax.text(j, i, f"{sim[i,j]:.2f}", ha="center", va="center", fontsize=7,
                color="white" if sim[i,j] > 0.6 else "black")
plt.colorbar(im, label="cosine similarity")
plt.title("Which documents 'know about' each other?")
plt.tight_layout(); plt.show()

print("Strongest non-identical pairs:")
pairs = [(sim[i,j], short_labels[i], short_labels[j])
         for i in range(len(docs)) for j in range(i+1, len(docs))]
for s, a, b in sorted(pairs, reverse=True)[:4]:
    print(f"  {s:.3f}  {a}  <->  {b}")

---
## Part C: Semantic search vs. keyword search, head to head

This is the payoff. We ask a question using words that appear **nowhere** in the target document, and compare what each method retrieves.

In [ ]:
def semantic_search(query, corpus, corpus_emb, top_n=3):
    q = embedder.encode([query])
    scores = cosine_similarity(q, corpus_emb)[0]
    idx = np.argsort(scores)[::-1][:top_n]
    return [(corpus[i], float(scores[i])) for i in idx]

def keyword_search(query, corpus, top_n=3):
    qwords = set(query.lower().replace('?','').replace('.','').split())
    scored = []
    for doc in corpus:
        dwords = set(doc.lower().replace('.','').replace('%','').split())
        overlap = len(qwords & dwords)
        scored.append((doc, overlap))
    scored.sort(key=lambda x: -x[1])
    return scored[:top_n]

QUERY = "How long until a new employee is productive?"

print(f'QUERY: "{QUERY}"')
print("(note: the words 'onboarding' and 'ramp-up' do not appear in the query)\n")

print("--- SEMANTIC SEARCH (embeddings) ---")
for doc, score in semantic_search(QUERY, docs, emb):
    print(f"  {score:.3f}  {doc}")

print("\n--- KEYWORD SEARCH (word overlap) ---")
for doc, score in keyword_search(QUERY, docs):
    print(f"  {score} words  {doc}")

## Cell 6 — ✏️ CHANGE ME: try to break it

Write queries against the same 8 documents. Try to find:
1. A query where semantic search clearly wins
2. A query where keyword search does fine (usually: exact names, IDs, error codes)
3. A query where semantic search retrieves something plausible but **wrong**

In [ ]:
# ✏️ CHANGE ME
MY_QUERIES = [
    "Why did profitability decline?",
    "Tuesday outage",
    "Did revenue go down?",          # <- watch this one carefully
]

for q in MY_QUERIES:
    print(f'QUERY: "{q}"')
    top = semantic_search(q, docs, emb, top_n=2)
    for doc, score in top:
        print(f"   {score:.3f}  {doc}")
    print()

### 💬 Discussion — Part C
- On `"Did revenue go down?"`, what does semantic search return? Is the top hit actually answering the question, or just talking about the same subject? *(This is the "fell vs. grew" problem from Part A, now inside a retrieval system.)*
- Where did keyword search hold its own? Most production systems use **both** (hybrid search) for exactly this reason — what does each contribute?
- Retrieval returns the *nearest* chunks, always. It has no notion of "nothing here answers this." What has to sit on top of retrieval to handle that? *(A guardrail: you build one in Session 24. Hold this thought.)*

---
## Part D: The chunk-size tradeoff

Before you retrieve, you must **chunk** — split long documents into pieces. Chunk size is the single most consequential parameter in a RAG system, and there's no universally right answer.

Below, the same document is chunked three ways and queried identically.

In [ ]:
report = (
    "Nimbus Bank reported net income of 412 million dollars for fiscal 2024. "
    "This represented an increase of 8 percent over the prior year. "
    "The improvement was driven primarily by higher net interest income. "
    "Deposit balances grew to 31 billion dollars during the same period. "
    "However, the bank recorded a 45 million dollar provision for credit losses. "
    "Management attributed the provision to weakness in commercial real estate. "
    "The efficiency ratio improved to 58 percent from 62 percent. "
    "Headcount declined by 3 percent following the branch consolidation program."
)

def chunk_text(text, chunk_chars, overlap=0):
    step = max(1, chunk_chars - overlap)
    return [text[i:i+chunk_chars] for i in range(0, len(text), step) if text[i:i+chunk_chars].strip()]

QUESTION = "Why did the bank record a provision for credit losses?"
print(f'QUESTION: "{QUESTION}"\n')

for size in [60, 150, 400]:
    chunks = chunk_text(report, size)
    cemb = embedder.encode(chunks)
    top = semantic_search(QUESTION, chunks, cemb, top_n=1)[0]
    print(f"--- chunk size {size} chars -> {len(chunks)} chunks ---")
    print(f"    best score: {top[1]:.3f}")
    print(f"    retrieved : \"{top[0].strip()}\"")
    print()

### 💬 Discussion — Part D
- At the smallest chunk size, does the retrieved text contain enough to actually *answer* the question, or just enough to match it?
- At the largest chunk size, the retrieved text almost certainly contains the answer — but what else does it drag along, and what does that cost you? *(Recall Session 12: everything retrieved gets pasted into the context window, and you pay per token.)*
- The middle size retrieves the two sentences that belong together. What does that suggest about aligning chunk boundaries with the document's own structure — sentences, paragraphs, sections — rather than a fixed character count?
- **Next session** you'll run this exact experiment on Apple's real 10-K. What's your prediction for a 100-page filing versus this 8-sentence paragraph?

## ✅ Notebook 17 complete — Reflection questions

These prep you for **Quiz 5** (Session 20).

1. What is an embedding, in one sentence a non-technical colleague would understand?
2. Explain why "revenue grew 12%" and "revenue fell 12%" score as highly similar, and why that's dangerous in retrieval.
3. Give one query type where keyword search beats semantic search.
4. Describe the chunk-size tradeoff in business terms — what do you lose at each extreme?
5. Retrieval always returns *something*. What must a production system add to handle "no relevant document exists"?

---
*Next session: you build the full pipeline yourself, on Apple's actual 10-K filing.*